## 1. `date_score()` → date_score, remove

Idea: every recipe has its own rhythm (mean `mean` and std `std` of the gaps between its last 20 `dates`).
The goal is to propose it again close to that rhythm, so its `std` gets small – plus a small shift that the Wdh rating (0–10) decides.

1. `mean`, `std` = mean / std of the gaps between the (max. 20) `dates`
2. `M` = rotation point = recipes cooked ≥ 2 times · days between app uses (last 30 days)
   e.g. 10 uses in a month → every 3 days; 10 recipes in rotation → `M` = 30 days
3. `w = tanh((5 − repeat) / k_r)` ∈ (−1, 1): Wdh 5 = no shift, 10 → earlier, 0 → later
4. target `T = mean + w · |M − mean|` (< 2 dates → `mean = M`)
5. `amp` so that the tanh reaches **90 %** of max at `T`, clipped to [1, 4]
6. `date_score = 10 · tanh(amp / tau · (dist_last − nulldurchgang))`, ≤ 0 → 0 and removed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Stellschrauben ----------------
nulldurchgang = 5        # bis hierhin (Tage seit last_date) ist der Score <= 0 -> removed
BASE_DAYS = 60           # amp = 1 (keine Verstaerkung) erreicht ...
BASE_LEVEL = 0.8         # ... 80 % vom Max nach BASE_DAYS Tagen
TARGET_LEVEL = 0.9       # beim Zielabstand T soll die Kurve 90 % vom Max erreichen
AMP_MIN, AMP_MAX = 1, 4  # Grenzen der amplification
MAX_SCORE = 10
NEVER_COOKED = 5         # date_score fuer nie gekochte Rezepte
REPEAT_CENTER = 5        # Wdh 0-10: hier keine Verschiebung
REPEAT_K = 2.5           # Steilheit der Wdh-Gewichtung (tanh)
EMPTY_REPEAT = 5         # Wdh ohne Bewertung -> neutral, keine Verschiebung
HISTORY_MAX = 20         # wie RatingLogic.HISTORY_ARRAY_MAX: nur die letzten 20 dates zaehlen
USAGE_WINDOW = 30        # Tage, ueber die die App-Nutzung gezaehlt wird ("ein Monat")
MIN_COOKS = 2            # ab so vielen dates zaehlt ein Rezept zur Rotation
FALLBACK_USAGE = 3       # Tage zwischen Nutzungen, falls im Fenster nichts gekocht wurde

tau = (BASE_DAYS - nulldurchgang) / np.arctanh(BASE_LEVEL)   # amp = 1 -> 80 % nach 60 Tagen
TODAY = pd.Timestamp("2026-10-13")


def to_days(dates):
    """ISO-Strings / Timestamps -> sortiertes Array in Tagen."""
    if len(dates) == 0:
        return np.array([])
    return np.sort(pd.to_datetime(list(dates)).values.astype("datetime64[D]").astype(float))


def interval_stats(dates):
    """mean und std der Abstaende zwischen den letzten HISTORY_MAX dates (gleicher Tag zaehlt
    einmal, sonst entstehen 0-Tage-Abstaende). < 2 dates -> (nan, nan)."""
    d = np.unique(to_days(dates)[-HISTORY_MAX:])
    if len(d) < 2:
        return np.nan, np.nan
    gaps = np.diff(d)
    return gaps.mean(), gaps.std()


def rotation_point(history, today=TODAY):
    """M = Rezepte in Rotation * Tage zwischen App-Nutzungen (im letzten Monat)."""
    t = to_days([today])[0]
    n_rotation = sum(len(h.get("dates", [])) >= MIN_COOKS for h in history)
    uses = sum(t - USAGE_WINDOW < d <= t for h in history for d in to_days(h.get("dates", [])))
    usage_every = USAGE_WINDOW / uses if uses else FALLBACK_USAGE
    return max(n_rotation, 1) * usage_every, n_rotation, usage_every


def repeat_weight(repeat):
    """Wdh 0-10 -> Anteil der Verschiebung in (-1, 1): 5 -> 0, 10 -> ~-1 (frueher), 0 -> ~+1 (spaeter)."""
    return np.tanh((REPEAT_CENTER - np.asarray(repeat, dtype=float)) / REPEAT_K)


def target_days(mean, M, repeat):
    """Zielabstand T = mean + w(repeat) * |M - mean|. Ohne mean (< 2 dates) gilt M."""
    mean = M if np.isnan(mean) else mean
    return mean + repeat_weight(repeat) * abs(M - mean)


def amplification(T):
    """Steilheit, bei der die Kurve bei T genau TARGET_LEVEL erreicht - begrenzt auf [AMP_MIN, AMP_MAX]."""
    gap = np.maximum(np.asarray(T, dtype=float) - nulldurchgang, 1e-9)
    return np.clip(tau * np.arctanh(TARGET_LEVEL) / gap, AMP_MIN, AMP_MAX)


def date_score_curve(amp, dist_last):
    return MAX_SCORE * np.tanh(amp / tau * (np.asarray(dist_last, dtype=float) - nulldurchgang))


def date_score(entry, M, today=TODAY):
    """date_score fuer EINEN history-Eintrag -> dict mit date_score, remove und allen Zwischenwerten."""
    if not entry or not entry.get("last_date"):
        return dict(date_score=NEVER_COOKED, remove=False)
    dist_last = (pd.Timestamp(today) - pd.Timestamp(entry["last_date"])).days
    mean, std = interval_stats(entry.get("dates", []))
    ratings = [r for r in entry.get("repeat_rating", []) if r is not None]
    repeat = float(np.mean(ratings)) if ratings else EMPTY_REPEAT
    T = target_days(mean, M, repeat)
    amp = float(amplification(T))
    raw = float(date_score_curve(amp, dist_last))
    T_90 = nulldurchgang + tau * np.arctanh(TARGET_LEVEL) / amp     # wo 90 % wirklich erreicht wird
    return dict(dist_last=dist_last, mean=mean, std=std, repeat=repeat, T=T, amp=amp, T_90=T_90,
                date_score=max(0.0, raw), remove=raw <= 0)


LEVELS = {0: "o", 0.5: "s", 0.8: "^", 0.9: "D"}   # Anteil vom Maximum : Marker


def plot_date_score(ax, amplification, dist_last, label=None, **kwargs):
    """Zeichnet eine date_score-Kurve und markiert Nulldurchgang, 50 %, 80 %, 90 %."""
    line, = ax.plot(dist_last, date_score_curve(amplification, dist_last), label=label, **kwargs)
    c = amplification / tau
    for p, m in LEVELS.items():
        x = nulldurchgang + np.arctanh(p) / c
        if x <= dist_last.max():
            ax.plot(x, p * MAX_SCORE, m, color=line.get_color(), ms=7)
    return line


def style_axes(ax, title):
    """Hilfslinien, Marker-Legende und Beschriftung."""
    for p, m in LEVELS.items():
        ax.plot([], [], m, color="black", ls="",
                label="Nulldurchgang" if p == 0 else f"{p:.0%} von max")
        ax.axhline(p * MAX_SCORE, color="gray", lw=0.5, ls=":")
    ax.axvline(nulldurchgang, color="gray", lw=0.8, ls="--")
    ax.set_xlabel("dist_last")
    ax.set_ylabel("date_score")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

### Base curve and amplification limits

`amp = 1` is the slowest curve (80 % after 60 days), `amp = 4` the fastest.
Because 90 % must be hit at `T`, the clip means: targets shorter than `T_min` are reached at `T_min`, longer than `T_max` at `T_max`.

In [ ]:
dist_last = np.linspace(0, 120, 500)
T_min = nulldurchgang + tau * np.arctanh(TARGET_LEVEL) / AMP_MAX
T_max = nulldurchgang + tau * np.arctanh(TARGET_LEVEL) / AMP_MIN
print(f"tau = {tau:.1f}   90 % reachable between T_min = {T_min:.1f} and T_max = {T_max:.1f} days")

fig, ax = plt.subplots(figsize=(8, 5))
for amp in [AMP_MIN, 2, 3, AMP_MAX]:
    plot_date_score(ax, amp, dist_last, label=f"amp={amp}")
style_axes(ax, "date_score for fixed amplification")
plt.tight_layout()
plt.show()

### Shift: Wdh rating → target `T`

Left: fraction `w` of the shift. Right: target `T` for some recipe rhythms `mean` at a fixed rotation point `M`.
Dashed: `T_min` / `T_max` where the amplification clip takes over.

In [ ]:
repeats = np.linspace(0, 10, 201)
M_demo = 30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(repeats, repeat_weight(repeats))
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].axvline(REPEAT_CENTER, color="gray", lw=0.8, ls="--")
axes[0].set_xlabel("Wdh (repeat_rating mean)")
axes[0].set_ylabel("w = share of |M - mean|")
axes[0].set_title(f"repeat_weight (k_r={REPEAT_K})")
axes[0].grid(alpha=0.3)

for mean in [7, 15, 30, 45, 70]:
    axes[1].plot(repeats, [target_days(mean, M_demo, r) for r in repeats], label=f"mean={mean}")
for y in [T_min, T_max]:
    axes[1].axhline(y, color="gray", lw=0.8, ls="--")
axes[1].set_xlabel("Wdh (repeat_rating mean)")
axes[1].set_ylabel("target T (days)")
axes[1].set_title(f"target T at M = {M_demo}")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Ts = np.linspace(0, 120, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(Ts, amplification(Ts))
for x in [T_min, T_max]:
    ax.axvline(x, color="gray", lw=0.8, ls="--")
ax.set_xlabel("target T (days)")
ax.set_ylabel("amplification")
ax.set_title(f"amp so that the curve is at {TARGET_LEVEL:.0%} at T, clipped to [{AMP_MIN}, {AMP_MAX}]")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Rotation point `M`

`M = n_rotation · usage_every`, `usage_every = 30 / uses in the last 30 days`.

In [ ]:
uses_per_month = [5, 10, 20, 30]
n_rotations = [5, 10, 20, 40]
table = pd.DataFrame(
    [[n * USAGE_WINDOW / u for u in uses_per_month] for n in n_rotations],
    index=pd.Index(n_rotations, name="recipes in rotation"),
    columns=pd.Index(uses_per_month, name="uses / month"),
)
table.round(1)

### Scenarios

Synthetic history (the real `history.yaml` has only one entry so far). `M` is computed from this history.
`T` = wanted target, `T_90` = where 90 % is really reached (differs when `amp` is clipped).

In [ ]:
rng = np.random.default_rng(0)


def cooked(every, n, last_ago, repeat, jitter=0):
    """history-Eintrag: n mal gekocht, im Abstand every (+- jitter), zuletzt vor last_ago Tagen."""
    ago = last_ago + np.cumsum([0] + [max(1, every + rng.integers(-jitter, jitter + 1)) for _ in range(n - 1)])
    dates = sorted((TODAY - pd.Timedelta(days=int(a))).date().isoformat() for a in ago)
    return {"dates": dates, "last_date": dates[-1], "repeat_rating": [repeat] * n}


def cluster(gaps, last_ago, repeat):
    """history-Eintrag mit fest vorgegebenen Abstaenden (aelteste zuerst)."""
    ago = last_ago + np.cumsum([0] + list(gaps[::-1]))
    dates = sorted((TODAY - pd.Timedelta(days=int(a))).date().isoformat() for a in ago)
    return {"dates": dates, "last_date": dates[-1], "repeat_rating": [repeat] * len(dates)}


scenarios = {
    "favourite, every 7 d, Wdh 9":   cooked(7, 12, 3, 9, jitter=1),
    "favourite, every 7 d, Wdh 9 (8 d ago)": cooked(7, 12, 8, 9, jitter=1),
    "regular, every 30 d, Wdh 5":    cooked(30, 6, 20, 5, jitter=3),
    "rare, every 60 d, Wdh 5":       cooked(60, 4, 40, 5, jitter=5),
    "rare, every 60 d, Wdh 9":       cooked(60, 4, 40, 9, jitter=5),
    "often, every 10 d, Wdh 1":      cooked(10, 8, 12, 1, jitter=2),
    "cluster + long pause, Wdh 5":   cluster([1, 1, 2, 70, 80], 25, 5),
    "cooked once, Wdh 7":            {"dates": ["2026-09-01"], "last_date": "2026-09-01", "repeat_rating": [7]},
    "just cooked (2 d), Wdh 10":     cooked(14, 5, 2, 10, jitter=2),
    "never cooked":                  {},
}

M, n_rotation, usage_every = rotation_point(scenarios.values())
print(f"recipes in rotation = {n_rotation}, app used every {usage_every:.1f} d  ->  M = {M:.1f} days")

rows = {name: date_score(entry, M) for name, entry in scenarios.items()}
pd.DataFrame(rows).T[["dist_last", "mean", "std", "repeat", "T", "amp", "T_90", "date_score", "remove"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, r in rows.items():
    if "T" not in r:
        continue
    line = plot_date_score(ax, r["amp"], dist_last, label=f"{name}: T={r['T']:.0f}, amp={r['amp']:.2f}")
    ax.plot(r["dist_last"], r["date_score"], "x", color=line.get_color(), ms=10, mew=2)
style_axes(ax, f"Scenarios (M = {M:.1f}); x = today")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()

## Sanity Checks

Expected behaviour; the cell below checks each row.

| Situation | Expected | Status |
|---|---|---|
| just cooked | 0, removed | 🟠 |
| cooked within `nulldurchgang` days (incl.) | removed | 🟠 |
| `nulldurchgang` + 1 days ago | not removed | 🟠 |
| never cooked | `NEVER_COOKED` (5) | 🟠 |
| at `T` (no clip) | 90 % of max | 🟠 |
| `dist_last` ≥ max(`T`, `T_min`) | ≥ 90 % of max | 🟠 |
| Wdh = 5 | `T = mean` (no shift) | 🟠 |
| `mean = M` | `T = M` for every Wdh | 🟠 |
| higher Wdh ⇒ earlier `T` ⇒ higher score | monotonically increasing | 🟠 |
| longer ago ⇒ higher score | monotonically increasing | 🟠 |
| < 2 dates | `T = M` | 🟠 |
| only the last 20 dates count | older dates ignored | 🟠 |
| `amp` | always in [1, 4] | 🟠 |
| `date_score` | always in [0, 10] | 🟠 |
| 10 uses in 30 d, 10 recipes cooked ≥ 2× | `M = 30` | 🟠 |

In [ ]:
def entry_every(every, n, last_ago, repeat):
    """Exakt regelmaessiger history-Eintrag (ohne jitter)."""
    ago = last_ago + every * np.arange(n)
    dates = sorted((TODAY - pd.Timedelta(days=int(a))).date().isoformat() for a in ago)
    return {"dates": dates, "last_date": dates[-1], "repeat_rating": [repeat] * n}


def score(every, last_ago, repeat=5, M=30, n=5):
    return date_score(entry_every(every, n, last_ago, repeat), M)


grid = [date_score(entry_every(e, n, a, r), M_)
        for e in [1, 3, 7, 14, 30, 60, 120] for n in [1, 2, 5, 20, 25]
        for a in [0, 3, 5, 6, 10, 30, 90, 200] for r in [0, 2.5, 5, 7.5, 10] for M_ in [3, 30, 120]]

new = entry_every(20, 20, 0, 5)                         # die letzten 20 dates im Abstand 20 ...
old = entry_every(2, 10, 20 * 19 + 2, 5)                # ... davor 10 aeltere im Abstand 2
merged = {**new, "dates": sorted(old["dates"] + new["dates"])}
ten_uses = [entry_every(30, 2, 3 * i, 5) for i in range(10)]   # 10 Rezepte, je 1x im letzten Monat

checks = [
    ("just cooked -> 0, removed",
     score(14, 0)["date_score"] == 0 and score(14, 0)["remove"]),
    (f"cooked within {nulldurchgang} days -> removed",
     all(score(14, a)["remove"] for a in range(nulldurchgang + 1))),
    (f"{nulldurchgang + 1} days ago -> not removed",
     not score(14, nulldurchgang + 1)["remove"]),
    ("never cooked -> NEVER_COOKED",
     date_score({}, 30)["date_score"] == NEVER_COOKED and not date_score({}, 30)["remove"]),
    ("at T (no clip) -> 90 %",
     np.isclose(score(40, 40)["date_score"], TARGET_LEVEL * MAX_SCORE)),
    ("dist_last >= max(T, T_min) -> >= 90 %",
     all(score(e, a)["date_score"] >= TARGET_LEVEL * MAX_SCORE - 1e-9
         for e in [7, 14, 30, 60, 120] for a in range(200) if a >= max(score(e, a)["T"], T_min))),
    ("Wdh 5 -> T = mean",
     all(np.isclose(score(e, 10, repeat=5)["T"], e) for e in [3, 7, 30, 90])),
    ("mean = M -> T = M for every Wdh",
     all(np.isclose(score(30, 10, repeat=r, M=30)["T"], 30) for r in range(11))),
    ("higher Wdh -> earlier T, higher score",
     all(score(40, 25, repeat=r)["T"] > score(40, 25, repeat=r + 1)["T"]
         and score(40, 25, repeat=r)["date_score"] <= score(40, 25, repeat=r + 1)["date_score"]
         for r in range(10))),
    ("longer ago -> higher score",
     all(score(30, a)["date_score"] <= score(30, a + 1)["date_score"] for a in range(150))),
    ("< 2 dates -> T = M",
     np.isclose(score(30, 10, n=1, M=42)["T"], 42)),
    ("only the last 20 dates count",
     np.isclose(date_score(merged, 30)["mean"], 20) and np.isclose(date_score(merged, 30)["std"], 0)),
    ("amp always in [1, 4]",
     all(AMP_MIN <= g["amp"] <= AMP_MAX for g in grid if "amp" in g)),
    ("date_score always in [0, 10]",
     all(0 <= g["date_score"] <= MAX_SCORE for g in grid)),
    ("10 uses in 30 d, 10 recipes cooked >= 2x -> M = 30",
     np.isclose(rotation_point(ten_uses)[0], 30)),
]

for name, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {name}")
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} passed")